In [ ]:
import sys, os
from pathlib import Path

parent_folder = str(Path.cwd().parents[2])
if parent_folder not in sys.path:
    sys.path.append(parent_folder)
from sigpy import mri
import scipy
import pickle
from sklearn.decomposition import PCA
from matplotlib.colors import ListedColormap
import seaborn as sns
import sigpy as sp
import cupy as cp
import numpy as np
from sigpy.mri.app import L1WaveletRecon


## My files
import save_data_helpers
import recon_functions
import recon_plot_helpers
from gating_functions import golden_angle_coords_3d
import sigpy.plot as pl

### Load all data

In [ ]:
ksp_512 = save_data_helpers.read_pickle('/home/lilianae/projects/naf_clean/load_data_clean/subject2_mid0082/ksp_from_mdb_512_samples.pkl')
ksp_512 = np.transpose(ksp_512, (2, 0, 1, 3))
print(f'ksp_512.shape = {ksp_512.shape}')

## Use first 400 spokes to speed up computation
ksp_data = ksp_512[:, :, :400, :]
print(f'ksp_data.shape = {ksp_data.shape}')

ncoils, nslices, nspokes, nsamples = ksp_data.shape
img_shape = (nslices, nsamples, nsamples)

## Golden angle coords
coords = golden_angle_coords_3d(img_shape=img_shape, num_spokes=nspokes, num_points=nsamples)
print(f'coords.shape = {coords.shape}')

espirit_mps = save_data_helpers.read_pickle('/home/lilianae/projects/naf_clean/coils/subject2_mid0082/espirit_mps_full_res_ksp_512_ungated.pkl')
print(f'espirit_mps.shape = {espirit_mps.shape}')

dcf_ksp = save_data_helpers.read_pickle('/home/lilianae/projects/naf_clean/recons/subject2_mid0082/dcf_ksp_less_spokes_512.pkl')
print(f'dcf_ksp.shape = {dcf_ksp.shape}')

ksp_with_dcf = ksp_data * dcf_ksp[None,:,:, :]
print(f'ksp_with_dcf.shape = {ksp_with_dcf.shape}')

In [ ]:
nz, ny, nx = espirit_mps[0].shape

S = sp.linop.Multiply((nz, ny, nx), espirit_mps)

print(f'Input shape of Sense adjoint operator = {S.H.ishape}')
print(f'Output shape of Sense adjoint operator = {S.H.oshape}')

In [ ]:
## Creat NUFFT Operator. 
## We will be applying adjoint:
## First argument is desired output shape, second arg is non-cartesian coordinate system

F = sp.linop.NUFFT((ncoils, nz, ny, nx), coord=coords)

print(f'Input shape of NUFFTAdjoint operator = {F.H.ishape}')
print(f'Output shape of NUFFTAdjoint operator = {F.H.oshape}')

## We want NUFFT for each coil, extend input dimension

### Forward op = Sense + NUFFT

In [ ]:
### Create forward op using NUFFT and Sense
A = F * S
A.repr_str = 'Sense'
print(f'A = {A}')

### Normalize this forward operator
# device = 0
# max_eig_op = sp.app.MaxEig(A.H * A, dtype=cp.complex64, device=device,max_iter=30).run()  
# A = (1/np.sqrt(max_eig_op))*A
# print(f'A = {A}')

### Assembled L1 wavelet recon

In [ ]:
# alg02 = L1WaveletRecon(ksp_data, lamda=1e-8,
#                        mps=espirit_mps,
#                        weights=None, coord=coords,
#                        transp_nufft=True, device=0)
# result2=alg02.run()
# output2=np.abs(cp.asnumpy(result2))

In [ ]:
wave_name='db4'
lamda = 1e-8
W = sp.linop.Wavelet(img_shape, wave_name=wave_name)
proxg = sp.prox.UnitaryTransform(sp.prox.L1Reg(W.oshape, lamda), W)
max_iter = 30
device=0 

def g(input):
    device = sp.get_device(input)
    xp = device.xp
    with device:
        return lamda * xp.sum(xp.abs(W(input))).item()

alg02 = sp.app.LinearLeastSquares(A,
                                  sp.to_device(ksp_with_dcf,device),
                                  proxg=proxg, g=g, 
                                  max_iter=max_iter)

result2=alg02.run()
output2=np.abs(cp.asnumpy(result2))